# COVID-19 Global Data Analysis

## Project 2 — Python & Pandas

### Objective
Analyse country-level COVID-19 data to identify differences in confirmed cases, recoveries, deaths, active cases and recent case growth.

### Key analytical questions
1. Which countries and WHO regions had the largest confirmed-case burden?
2. Which countries had the largest numbers of deaths and active cases?
3. How did recovery and mortality rates differ?
4. Which countries showed the fastest recent growth?
5. What relationships existed between confirmed cases, deaths, recoveries and active cases?
6. What can these metrics tell us — and what can they **not** tell us?

> **Important:** This project analyses reported figures in the supplied dataset. The analysis does not establish causation or explain healthcare-system performance on its own.

## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

df = pd.read_excel("country_wise_latest.csv.xlsx")

## 2. Initial data inspection

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.describe()

### Data-quality checks

The dataset contains **187 rows and 16 columns**. The original notebook checks for missing values and duplicate rows; both checks returned zero.

The dataset includes country/region identifiers, absolute COVID-19 measures, percentage measures, one-week changes and WHO regional classifications.

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## 3. Confirmed cases by country

In [ ]:
top10_confirmed = (
    df.sort_values("Confirmed", ascending=False)
      .head(10)
)

top10_confirmed[["Country/Region", "Confirmed"]]

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(top10_confirmed["Country/Region"], top10_confirmed["Confirmed"])
plt.title("Top 10 Countries by Confirmed COVID-19 Cases")
plt.xlabel("Country")
plt.ylabel("Confirmed Cases")
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

### Interpretation
The United States had the largest number of confirmed cases in the dataset, followed by Brazil and India. These are **absolute counts**, so they describe the scale of reported cases rather than the proportion of a country's population affected.

## 4. Confirmed cases by WHO region

In [ ]:
region_cases = (
    df.groupby("WHO Region")["Confirmed"]
      .sum()
      .sort_values(ascending=False)
)

print(region_cases)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(region_cases.index, region_cases.values)
plt.title("Confirmed COVID-19 Cases by WHO Region")
plt.xlabel("WHO Region")
plt.ylabel("Confirmed Cases")
plt.xticks(rotation=20)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

### Key finding
The **Americas had the highest total confirmed-case burden in the dataset**. This is an observation about reported cases; the dataset alone cannot determine why the region had the highest burden.

## 5. Deaths by country

In [ ]:
top10_deaths = (
    df.sort_values("Deaths", ascending=False)
      .head(10)
)

top10_deaths[["Country/Region", "Deaths"]]

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top10_deaths["Country/Region"], top10_deaths["Deaths"])
plt.title("Top 10 Countries by COVID-19 Deaths")
plt.xlabel("Deaths")
plt.ylabel("Country")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.gca().invert_yaxis()
plt.show()

## 6. Recovery rate

In [ ]:
df["Recovery Rate"] = (df["Recovered"] / df["Confirmed"]) * 100

df[["Country/Region", "Confirmed", "Recovered", "Recovery Rate"]].head()

In [ ]:
recovery_filtered = (
    df[df["Confirmed"] >= 10000]
      .sort_values("Recovered / 100 Cases", ascending=False)
)

recovery_filtered[
    ["Country/Region", "Confirmed", "Recovered",
     "Recovered / 100 Cases", "Deaths / 100 Cases"]
].head(10)

### Interpretation
Filtering to countries with at least **10,000 confirmed cases** reduces the influence of very small case counts when comparing recovery proportions. The recovery percentage should still be interpreted as a reported dataset measure, not as a complete clinical recovery probability.

## 7. Mortality rate

In [ ]:
df["Death Rate"] = (df["Deaths"] / df["Confirmed"]) * 100

death_rate_by_region = (
    df.groupby("WHO Region")["Death Rate"]
      .mean()
      .sort_values(ascending=False)
)

print(death_rate_by_region)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(death_rate_by_region.index, death_rate_by_region.values)
plt.title("Average Reported Death Rate by WHO Region")
plt.xlabel("WHO Region")
plt.ylabel("Average Death Rate (%)")
plt.xticks(rotation=20)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

In [ ]:
death_filtered = (
    df[df["Confirmed"] >= 10000]
      .sort_values("Deaths / 100 Cases", ascending=False)
)

death_filtered[
    ["Country/Region", "Confirmed", "Deaths", "Deaths / 100 Cases"]
].head(10)

### Key finding
Among countries with at least **10,000 confirmed cases**, the United Kingdom had the highest reported `Deaths / 100 Cases` value in this dataset at **15.19%**.

This should **not** be interpreted as proof that the UK had the worst healthcare system. Testing, case detection, population characteristics, timing and reporting practices are not captured here.

## 8. Active cases and current burden

In [ ]:
region_active = (
    df.groupby("WHO Region")["Active"]
      .sum()
      .sort_values(ascending=False)
)

print(region_active)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(region_active.index, region_active.values)
plt.title("Active COVID-19 Cases by WHO Region")
plt.xlabel("WHO Region")
plt.ylabel("Active Cases")
plt.xticks(rotation=20)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

In [ ]:
df["Active / 100 Cases"] = (df["Active"] / df["Confirmed"]) * 100

active_filtered = (
    df[df["Confirmed"] >= 10000]
      .sort_values("Active / 100 Cases", ascending=False)
)

active_filtered[
    ["Country/Region", "Confirmed", "Active", "Active / 100 Cases"]
].head(10)

### Key finding: absolute vs relative burden
The Serbia–US comparison demonstrates why both measures matter. Serbia had a much higher active-case percentage (about **97.75%**) while the US had a vastly larger absolute number of active cases (about **2.82 million**).

Therefore:
- **Active-case percentage** describes the relative share of confirmed cases that remained active.
- **Absolute active cases** describe the scale of the current reported case burden.

Neither metric should be used in isolation.

## 9. One-week case growth

In [ ]:
weekly_change = df.sort_values("1 week change", ascending=False)

weekly_change[
    ["Country/Region", "Confirmed last week", "Confirmed",
     "1 week change", "1 week % increase"]
].head(10)

In [ ]:
top10_confirmed_growth = (
    df.sort_values("Confirmed", ascending=False)
      .head(10)
      .sort_values("1 week % increase", ascending=False)
)

plt.figure(figsize=(10, 6))
plt.bar(
    top10_confirmed_growth["Country/Region"],
    top10_confirmed_growth["1 week % increase"]
)
plt.title("1-Week Percentage Increase Among Top 10 Countries by Confirmed Cases")
plt.xlabel("Country")
plt.ylabel("1 Week % Increase")
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

### Key finding
India recorded a **28.11% one-week increase** and was the highest among the **top 10 countries ranked by confirmed cases** in this comparison.

This wording is important: it does **not** mean India had the highest one-week percentage increase across every country in the dataset.

## 10. Relationship between confirmed cases and deaths

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["Confirmed"], df["Deaths"], alpha=0.7)
plt.title("Confirmed Cases vs Deaths")
plt.xlabel("Confirmed Cases")
plt.ylabel("Deaths")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

correlation = df["Confirmed"].corr(df["Deaths"])
print("Correlation:", correlation)

In [ ]:
correlation_matrix = df[
    ["Confirmed", "Deaths", "Recovered", "Active", "New cases", "New deaths"]
].corr()

print(correlation_matrix)

In [ ]:
plt.figure(figsize=(10, 7))
plt.imshow(correlation_matrix, interpolation="nearest")
plt.colorbar(label="Correlation")
plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45,
    ha="right"
)
plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns
)
plt.title("Correlation Matrix of COVID-19 Variables")
plt.show()

### Key finding
Confirmed cases and deaths showed a **very strong positive correlation (approximately 0.93)**. Countries with more confirmed cases generally also had more reported deaths.

However, **correlation does not establish causation**, and confirmed cases alone do not explain differences in mortality.

## 11. Key findings

1. **Regional burden:** The Americas had the highest total confirmed-case burden in the dataset.
2. **Confirmed cases and deaths:** Confirmed cases had a very strong positive correlation with deaths (approximately `r = 0.93`), but this does not prove causation.
3. **Recent growth:** India recorded a 28.11% one-week increase and had the highest weekly percentage increase among the top 10 countries by confirmed cases.
4. **Active-case interpretation:** Serbia had a much higher active-case percentage than the US, while the US had vastly more absolute active cases.
5. **Relative mortality:** Among countries with at least 10,000 confirmed cases, the UK had the highest reported deaths-per-100-cases value at 15.19%.
6. **Absolute vs relative metrics:** Total deaths and death rate answer different questions; absolute counts describe scale, while rates describe relative proportions.

## 12. Limitations

- **Confirmed cases are not necessarily all infections.** Testing availability, case detection and reporting practices can differ between countries.
- **Correlation does not prove causation.** The strong relationship between confirmed cases and deaths does not show that confirmed cases alone caused differences in deaths.
- **Important contextual variables are missing.** The dataset does not include factors such as population size, age distribution, healthcare capacity, testing rates or government interventions.
- **The data represent a particular point in time.** COVID-19 conditions changed rapidly, so these results should be interpreted as a snapshot rather than a complete description of the pandemic.
- **Percentage metrics can be misleading at small case counts.** This is why thresholds such as 10,000 confirmed cases were used for some rate comparisons.

## 13. Recommendations

1. **Targeted containment:** Prioritize containment measures in regions with high absolute active cases or rapid case growth, using additional local data to identify areas requiring intervention.
2. **Travel measures:** Consider targeted travel and border-control measures during periods of rapid international transmission, supported by epidemiological and travel data.
3. **Healthcare capacity:** Increase healthcare resources and capacity in areas experiencing high active-case burdens to help manage demand and reduce pressure on healthcare systems.

## 14. Conclusion

This analysis demonstrates that the COVID-19 burden varied considerably across countries and regions, highlighting the importance of using data to identify areas requiring attention and appropriate interventions. A strong positive correlation was observed between confirmed cases and deaths, although confirmed cases alone cannot explain differences in mortality. The analysis of active cases demonstrated the importance of considering both relative measures, such as active-case percentage, and absolute measures, such as the total number of active cases. Similarly, mortality rates should be interpreted carefully because they can be influenced by several factors beyond the number of confirmed cases. Overall, this project demonstrated that data analysis can provide valuable evidence for decision-making, but additional variables and contextual information are often required to fully understand and validate a finding.

## 15. Analyst takeaway

The main analytical lesson from this project is:

> **A metric is only useful when we understand what it measures, what it does not measure, and the context in which it should be interpreted.**

Throughout this project, absolute counts, percentages, rates and correlations were used for different questions rather than treating one metric as sufficient for every decision.